In [ ]:
import pandas as pd

In [24]:
event_df = pd.read_csv("../event.csv")
home_df  = pd.read_csv("../home_team.csv")
away_df  = pd.read_csv("../away_team.csv")

event_df = event_df[["match_id", "winner_code"]].copy()
home_df  = home_df[["match_id", "player_id", "full_name", "current_rank"]].copy()
away_df  = away_df[["match_id", "player_id", "full_name", "current_rank"]].copy()

In [25]:
event_df = event_df.dropna(subset=["match_id", "winner_code"])
home_df  = home_df.dropna(subset=["match_id", "player_id", "current_rank"])
away_df  = away_df.dropna(subset=["match_id", "player_id", "current_rank"])

In [26]:
before = len(event_df)
event_df = event_df[event_df["winner_code"].isin([1, 2])]
print(f"[Info] Dropped {before - len(event_df):,} rows with invalid winner_code")

[Info] Dropped 0 rows with invalid winner_code


In [27]:
event_df = event_df.drop_duplicates(subset=["match_id"])
home_df  = home_df.drop_duplicates(subset=["match_id"])
away_df  = away_df.drop_duplicates(subset=["match_id"])

In [28]:
df = event_df.merge(
        home_df.rename(columns={
            "player_id"   : "home_player_id",
            "full_name"   : "home_full_name",
            "current_rank": "home_rank"}),
        on="match_id", how="inner"
     ).merge(
        away_df.rename(columns={
            "player_id"   : "away_player_id",
            "full_name"   : "away_full_name",
            "current_rank": "away_rank"}),
        on="match_id", how="inner")

In [29]:
before = len(df)
df = df[df["home_player_id"] != df["away_player_id"]]
print(f"[Info] Dropped {before - len(df):,} matches where home == away player")

[Info] Dropped 0 matches where home == away player


In [30]:
home_view = df[["match_id", "winner_code",
                "home_player_id", "home_full_name", "home_rank",
                "away_player_id", "away_rank"]].copy()
home_view.columns = ["match_id", "winner_code",
                     "player_id", "full_name", "current_rank",
                     "opponent_id", "opponent_rank"]
home_view["won"] = (home_view["winner_code"] == 1).astype(int)

away_view = df[["match_id", "winner_code",
                "away_player_id", "away_full_name", "away_rank",
                "home_player_id", "home_rank"]].copy()
away_view.columns = ["match_id", "winner_code",
                     "player_id", "full_name", "current_rank",
                     "opponent_id", "opponent_rank"]
away_view["won"] = (away_view["winner_code"] == 2).astype(int)

all_matches = pd.concat([home_view, away_view], ignore_index=True)

In [31]:
top10_matches = all_matches[all_matches["opponent_rank"] <= 10].copy()
print(f"\n[Info] Total matches vs top 10 opponents: {len(top10_matches):,}")


[Info] Total matches vs top 10 opponents: 232


In [32]:
top10_matches["full_name"] = top10_matches["full_name"].fillna(
    "Player_" + top10_matches["player_id"].astype(str))

In [56]:
stats = ( top10_matches
    .groupby(["player_id", "full_name"])
    .agg(
        current_rank     = ("current_rank",  "min"),
        matches_vs_top10 = ("won",           "count"),
        wins_vs_top10    = ("won",           "sum"))
    .reset_index())

stats["win_pct"] = (
    (stats["wins_vs_top10"] / stats["matches_vs_top10"] * 100)
    .round(2))

In [57]:
MIN_MATCHES = 3
before = len(stats)
stats = stats[stats["matches_vs_top10"] >= MIN_MATCHES]
print(f"[Info] Removed {before - len(stats):,} players with fewer than "
      f"{MIN_MATCHES} matches vs top 10\n")

[Info] Removed 103 players with fewer than 3 matches vs top 10



In [58]:
stats = stats.sort_values("win_pct", ascending=False).reset_index(drop=True)

stats.columns = ["Player ID", "Full Name", "Current Rank",
                 "Matches vs Top 10", "Wins vs Top 10", "Win %"]

print("-" * 70)
print("  Highest Winning % vs Top 10 Ranked Opponents")
print(f"  (minimum {MIN_MATCHES} matches)")
print("-" * 70)
print(stats.head(20).to_string(index=False))

----------------------------------------------------------------------
  Highest Winning % vs Top 10 Ranked Opponents
  (minimum 3 matches)
----------------------------------------------------------------------
 Player ID              Full Name  Current Rank  Matches vs Top 10  Wins vs Top 10  Win %
    185388           Humbert, Ugo          14.0                  3               3 100.00
    228272           Swiatek, Iga           1.0                  3               3 100.00
    179146       Kalinskaya, Anna          24.0                  5               4  80.00
    275923        Alcaraz, Carlos           2.0                  4               3  75.00
    206570         Sinner, Jannik           2.0                  3               2  66.67
     89632         Jarry, Nicolas          19.0                  3               2  66.67
     19728        Cirstea, Sorana          22.0                  3               2  66.67
    230056         Kostyuk, Marta          20.0                  3   